# PS1 - can the OOS label be sharpened from silver?

The current PS1 event is "an OOS Set episode". That treats a failed gate and a parked bus
validator identically, which is the most likely reason VALIDATOR scores AUC 0.997 (persistence,
not prediction) while GATE and TVM sit near 0.89.

`silver.dim_event_matrix` - Cubic's own Device Event Matrix - records, per event code, whether
that OOS **counts toward that fleet's Ventra availability KPI**. That is the client's own
definition of out-of-service, and nothing in this repo uses it as a label filter. This notebook
measures what it, and four other silver signals, do to the base rate and therefore to the AUC
the contract demands.

**Read-only.** No Aurora, no Spark, no writes. Large tables are inspected by parquet footer
metadata only, never loaded.

| # | Source | What it adds |
|---|---|---|
| E1 | `dim_event_matrix.oos_counted_{gate,bus,fmvd}_kpi` | Cubic's own KPI counting rule |
| E2 | `event_priority` (already on the episode fact) | 1=critical .. 4=info |
| E5 | `contains_commanded_oos_signal` (already on the fact) | drops commanded / maintenance OOS |
| E3 | `device_failures` / `device_outage` | chargeability - **TVM/GATE only** |
| E4 | `kpi_avail_enriched` relief columns | downtime Cubic itself excused |

**Two traps this notebook is built to avoid.** The episode fact's `event_type_id` is a *string*
and the matrix key is an *int*; mapping across that mismatch returns all-NaN and silently filters
to zero rows. And the silver exports live in the **gold** bucket, not artifacts. Both are checked
explicitly, and join coverage is printed before any number is trusted.

In [ ]:
# CELL 1 : configuration
import pandas as pd, numpy as np, s3fs, pyarrow.parquet as pq
from math import sqrt
from statistics import NormalDist

EPISODE_BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
EPISODE_PREFIX = "chicago/ps3_outputs/ps3_device_episode_fact"

# Silver exports are written by notebooks/export_silver_to_s3.py; the bucket and
# prefix are widgets, so CELL 3 discovers them rather than assuming.
CANDIDATE_BUCKETS = [
    "cubic-mars-pm-s3-datalake-dev-gold-170202974600",
    "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600",
    "cubic-mars-pm-s3-datalake-dev-raw-170202974600",
]

ELIGIBLE_DEVICES = {"GATE": 821, "TVM": 468, "VALIDATOR": 3028}   # PS2 label recount
MEASURED_AUC     = {"GATE": 0.888, "TVM": 0.894, "VALIDATOR": 0.997}
KPI_COL_FOR      = {"GATE": "oos_counted_gate_kpi",
                    "TVM": "oos_counted_fmvd_kpi",
                    "VALIDATOR": "oos_counted_bus_kpi"}
N = NormalDist()

def auc_required(p, precision=0.90, recall=0.80):
    """Minimum ROC-AUC for the contract pair at base rate p."""
    if not (0 < p < 1):
        return np.nan, np.nan
    fpr = recall * p * (1 - precision) / (precision * (1 - p))
    if fpr >= 1:
        return fpr, 0.5
    return fpr, N.cdf((N.inv_cdf(recall) - N.inv_cdf(fpr)) / sqrt(2))

for _p, _exp in ((0.039, 0.994), (0.078, 0.990), (0.157, 0.982), (0.624, 0.909)):
    assert abs(auc_required(_p)[1] - _exp) < 0.002, _p

fs = s3fs.S3FileSystem()
print("config loaded; auc_required maths OK")

In [ ]:
# CELL 2 : helpers
def parts_of(prefix):
    """Parquet part files under a prefix (newest computed_date if partitioned)."""
    try:
        p = sorted(f for f in fs.find(prefix) if f.endswith(".parquet"))
    except FileNotFoundError:
        return []
    if not p:
        return []
    if any("computed_date=" in x for x in p):
        newest = sorted({x.split("computed_date=")[1].split("/")[0]
                         for x in p if "computed_date=" in x})[-1]
        p = [x for x in p if f"computed_date={newest}" in x]
    return p

def footer(prefix):
    """(rows, columns) from parquet metadata. No data is read."""
    p = parts_of(prefix)
    if not p:
        return None, []
    rows, cols = 0, []
    for f in p:
        with fs.open(f, "rb") as fh:
            pf = pq.ParquetFile(fh)
            rows += pf.metadata.num_rows
            if not cols:
                cols = list(pf.schema_arrow.names)
    return rows, cols

def load(prefix, columns=None, max_rows=8_000_000):
    """Read into pandas. Refuses oversized tables unless columns are narrowed."""
    rows, _ = footer(prefix)
    if rows is None:
        raise FileNotFoundError(prefix)
    if rows > max_rows and columns is None:
        raise MemoryError(f"{prefix}: {rows:,} rows -- pass columns= to narrow")
    frames = []
    for f in parts_of(prefix):
        with fs.open(f, "rb") as fh:      # file handle, not path: no hive inference,
            frames.append(pd.read_parquet(fh, columns=columns))   # so run_id cannot clash
    return pd.concat(frames, ignore_index=True)

def flag(s):
    """Object/boolean column -> clean bool, NaN treated as False, no FutureWarning."""
    return s.eq(True)

print("helpers ready")

In [ ]:
# CELL 3 : locate the silver exports
SILVER_ROOT = None
for b in CANDIDATE_BUCKETS:
    try:
        top = fs.ls(b)
    except Exception as exc:
        print(f"  {b:56s} unreadable ({type(exc).__name__})"); continue
    print(f"  {b:56s} {len(top)} top-level prefixes")
    for t in top:
        try:
            subs = fs.ls(t)
        except Exception:
            continue
        for s in subs:
            if s.rstrip("/").rsplit("/", 1)[-1].lower() == "silver":
                n = len(fs.ls(s))
                print(f"      -> s3://{s}   ({n} tables)")
                if SILVER_ROOT is None and n > 5:
                    SILVER_ROOT = s          # "<bucket>/<prefix>/silver"

if SILVER_ROOT is None:
    raise SystemExit("no silver export found -- paste this output")
print(f"\nSILVER_ROOT = s3://{SILVER_ROOT}")
SILVER_TABLES = sorted(x.rstrip("/").rsplit("/", 1)[-1] for x in fs.ls(SILVER_ROOT))
print(f"{len(SILVER_TABLES)} tables: " + ", ".join(SILVER_TABLES))

In [ ]:
# CELL 4 : the episode fact -- what does it actually carry?
ep = load(f"{EPISODE_BUCKET}/{EPISODE_PREFIX}")
ep["episode_start"] = pd.to_datetime(ep["episode_start"])
WIN_DAYS = max(1, (ep["episode_start"].max() - ep["episode_start"].min()).days + 1)

print(f"episodes = {len(ep):,}   window {ep['episode_start'].min().date()} "
      f"-> {ep['episode_start'].max().date()}  ({WIN_DAYS} days)")
print("fleets:", dict(ep["mars_device_category"].value_counts()))

WANT = ["event_type_id", "event_priority", "contains_commanded_oos_signal",
        "chargeability_policy", "oos_fact_definition", "oos_minutes_union",
        "set_signal_span_minutes", "oos_set_event_count", "requires_service_call",
        "device_id", "component_subsystem",
        "is_chargeable", "failure_level", "transit_day", "availability_event_id"]
print("\nCOLUMNS THIS NOTEBOOK NEEDS:")
for c in WANT:
    print(f"  {'YES' if c in ep.columns else ' - ':4s} {c:34s}"
          + (f" {str(ep[c].dtype):16s} non-null {ep[c].notna().mean():6.1%}" if c in ep.columns else ""))

for c in ("chargeability_policy", "oos_fact_definition"):
    if c in ep.columns:
        print(f"\n{c}: {dict(ep[c].value_counts())}")

In [ ]:
# CELL 5 : silver inventory -- metadata only, nothing large is loaded
CANDIDATES = ["dim_event_matrix", "dim_event_type", "dim_device", "dim_failure_level",
              "device_failures", "device_outage", "device_event_enriched",
              "kpi_avail_enriched", "kpi_daily", "maintenance_ledger",
              "incident_history", "incident_root_cause", "incident_task_ci_link",
              "device_mttr", "device_survival_intervals", "warnings_daily"]
inv = []
for t in CANDIDATES:
    try:
        rows, cols = footer(f"{SILVER_ROOT}/{t}")
        inv.append(dict(table=t, rows=rows, n_cols=len(cols), cols=cols,
                        reach="pandas" if (rows or 0) <= 8_000_000 else "NEEDS SPARK"))
    except Exception as exc:
        inv.append(dict(table=t, rows=None, n_cols=0, cols=[],
                        reach=f"MISSING ({type(exc).__name__})"))
invdf = pd.DataFrame(inv)
print(invdf[["table", "rows", "n_cols", "reach"]].to_string(index=False))

In [ ]:
# CELL 6 : the event matrix -- Cubic's own OOS counting rule
mtx = load(f"{SILVER_ROOT}/dim_event_matrix")
print(f"dim_event_matrix: {len(mtx):,} event codes\n")
for c in [c for c in mtx.columns if c.startswith("oos_counted")
          or c in ("is_oos", "is_commanded_oos", "requires_service_call",
                   "event_priority", "is_reader_event")]:
    s = mtx[c]
    if set(pd.Series(s).dropna().unique()) <= {True, False, 0, 1}:
        print(f"  {c:28s} TRUE on {int(flag(s).sum()):4d} / {len(mtx)} codes")
    else:
        print(f"  {c:28s} {dict(s.value_counts().sort_index())}")

# --- the join, with the string/int trap handled explicitly --------------------
KEY = "event_code_id" if "event_code_id" in mtx.columns else mtx.columns[0]
print(f"\nmatrix key   : {KEY} ({mtx[KEY].dtype})")
print(f"episode key  : event_type_id ({ep['event_type_id'].dtype})")

m = mtx.copy()
m["_k"] = pd.to_numeric(m[KEY], errors="coerce").astype("Int64")
m = m.dropna(subset=["_k"]).drop_duplicates("_k").set_index("_k")
ep["_k"] = pd.to_numeric(ep["event_type_id"], errors="coerce").astype("Int64")

COVERAGE = ep["_k"].isin(m.index).mean()
print(f"\nJOIN COVERAGE: {COVERAGE:.1%} of episodes match a matrix code")
if COVERAGE < 0.995:
    print("  unmatched episode codes (top 15) -- raise with Michael if material:")
    print(ep.loc[~ep["_k"].isin(m.index), "event_type_id"]
            .value_counts().head(15).to_string())
assert COVERAGE > 0.05, "join produced almost nothing -- do not trust cell 7"

for col in [c for c in m.columns if c.startswith("oos_counted")]:
    ep[f"_mx_{col}"] = ep["_k"].map(m[col]).eq(True)
print("\nmatrix flags attached:", [c for c in ep.columns if c.startswith("_mx_")])

In [ ]:
# CELL 7 : measure the label under each enrichment
dur  = (pd.to_numeric(ep.get("oos_minutes_union"), errors="coerce")
          .fillna(pd.to_numeric(ep.get("set_signal_span_minutes"), errors="coerce"))
          .fillna(0.0))
prio = pd.to_numeric(ep["event_priority"], errors="coerce")
cmd  = flag(ep["contains_commanded_oos_signal"]) if "contains_commanded_oos_signal" in ep.columns \
       else pd.Series(False, index=ep.index)

# fleet-specific KPI rule collapsed into one column
kpi = pd.Series(False, index=ep.index)
for fl, col in KPI_COL_FOR.items():
    sel, mc = ep["mars_device_category"].eq(fl), f"_mx_{col}"
    if mc in ep.columns:
        kpi.loc[sel] = ep.loc[sel, mc]

VARIANTS = {
    "E0  every OOS Set episode (today)": pd.Series(True, index=ep.index),
    "B   duration >= 60 min":            dur >= 60,
    "E1  counts toward Ventra KPI":      kpi,
    "E2  event_priority <= 2":           prio.le(2).fillna(False),
    "E5  NOT commanded / maintenance":   ~cmd,
    "E1+E2  KPI-counted AND prio<=2":    kpi & prio.le(2).fillna(False),
    "E1+E5  KPI-counted, not commanded": kpi & ~cmd,
    "E1+B   KPI-counted AND >=60 min":   kpi & (dur >= 60),
}

rows = []
for fleet, g in ep.groupby("mars_device_category"):
    ndev = ELIGIBLE_DEVICES.get(fleet)
    if not ndev:
        continue
    device_days = ndev * WIN_DAYS
    for name, mask in VARIANTS.items():
        n = int(mask.loc[g.index].sum())
        for h in (3, 7):
            p = min(0.999, n * h / device_days)
            fpr, auc = auc_required(p)
            rows.append(dict(fleet=fleet, variant=name, episodes=n,
                             kept=n / max(1, len(g)), horizon=h, base_rate=p,
                             max_fpr=fpr, auc_required=auc,
                             auc_today=MEASURED_AUC[fleet],
                             gap=auc - MEASURED_AUC[fleet]))

res = pd.DataFrame(rows)
pd.set_option("display.width", 210); pd.set_option("display.max_rows", 400)
for fleet in sorted(res["fleet"].unique()):
    print("=" * 102)
    print(f"{fleet}   eligible={ELIGIBLE_DEVICES[fleet]:,}  "
          f"device-days={ELIGIBLE_DEVICES[fleet]*WIN_DAYS:,}  AUC today={MEASURED_AUC[fleet]}")
    print("=" * 102)
    print(res[res["fleet"] == fleet][["variant", "episodes", "kept", "horizon",
          "base_rate", "auc_required", "gap"]].round(4).to_string(index=False))
    print()
res.to_csv("ps1_label_enrichment_probe.csv", index=False)
print("saved -> ps1_label_enrichment_probe.csv")

In [ ]:
# CELL 8 : E3 / E4 feasibility -- chargeability and relief
# These are NOT on the episode fact. Can they be reached by joining silver?
def probe(table, wanted):
    try:
        rows, cols = footer(f"{SILVER_ROOT}/{table}")
    except Exception as exc:
        print(f"  {table:28s} MISSING ({type(exc).__name__})"); return None
    hit = [c for c in cols if any(w.lower() in c.lower() for w in wanted)]
    print(f"  {table:28s} {rows:>12,} rows  matching cols: {hit or 'NONE'}")
    return cols

print("E3 chargeability:")
probe("device_failures",     ["charge", "failure_level", "device_key", "failure_date", "source"])
probe("device_outage",       ["charge", "failure_level", "device_id", "transit_day"])
probe("incident_root_cause", ["charge", "failure_level", "device_id"])
print("\nE4 relief (downtime Cubic excused):")
probe("kpi_avail_enriched",  ["relief", "device_id", "failure_level", "excluded"])
print("\nE5 maintenance windows:")
probe("maintenance_ledger",  ["ledger_type", "device_id", "event_dtm", "duration"])

print("\nNOTE: device_failures is keyed on DEVICE_KEY, the episode fact on device_id.")
print("A join needs dim_device (DEVICE_KEY -> DEVICE_ID, is_current) as a bridge.")
print("Per the silver headers, chargeability exists for TVM/GATE only -- VALIDATOR")
print("has no ServiceNow incidents at all, so E3/E4 cannot cover it.")

In [ ]:
# CELL 9 : verdict
ok   = [r for r in inv if isinstance(r["rows"], int) and r["rows"] <= 8_000_000]
big  = [r for r in inv if isinstance(r["rows"], int) and r["rows"] > 8_000_000]
gone = [r for r in inv if not isinstance(r["rows"], int)]
print("REACHABLE FROM THIS NOTEBOOK")
print(f"  pandas          : {len(ok):2d}  {', '.join(r['table'] for r in ok)}")
print(f"  needs Spark     : {len(big):2d}  {', '.join(r['table'] for r in big)}")
print(f"  missing from S3 : {len(gone):2d}  {', '.join(r['table'] for r in gone)}")
print(f"\nmatrix join coverage: {COVERAGE:.1%}")

d = res[(res["horizon"] == 7) & res["gap"].notna() & (res["base_rate"] < 0.30)]
if len(d):
    print("\nNON-DEGENERATE options at a 7-day horizon, smallest AUC gap first:")
    print(d.sort_values("gap").groupby("fleet").head(3)
           [["fleet", "variant", "episodes", "base_rate", "auc_required", "gap"]]
           .round(4).to_string(index=False))
print("\nBase rates above ~0.30 are excluded above: at that density the event is")
print("no longer worth predicting, and the low AUC it 'requires' is an artifact.")
print("\nA sharper label LOWERS the base rate and so RAISES the AUC required. The")
print("reason to adopt E1 anyway is that it is the event Cubic itself measures.")

---

## E3 chargeability and E4 relief

Neither is on the episode fact, so both need a join. Cell 8 measured that
`incident_root_cause` (374,740 rows) and `kpi_avail_enriched` (779,566) are small enough for
pandas and key on `device_id` directly - no Spark and no `DEVICE_KEY` bridge.

The join is **device-day**: an episode is chargeable if a chargeable incident exists for that
device on that calendar day, and relieved if a relief record or `EXCLUDED` flag covers it.

Two things to read carefully in the output below. First, **match rate** - an episode with no
incident row at all is scored not-chargeable, which for VALIDATOR means everything, because
ServiceNow does not log bus validators. That zero is a coverage fact, not a finding. Second,
the **column choices** - the date and key columns are auto-detected and printed; if the choice
looks wrong, override `IRC_KEY` / `IRC_DATE` / `KAE_KEY` / `KAE_DATE` and re-run.

In [ ]:
# CELL 10 : schemas of the chargeability / relief sources
def describe(table):
    rows, cols = footer(f"{SILVER_ROOT}/{table}")
    print(f"\n=== {table}  ({rows:,} rows, {len(cols)} cols) ===")
    print("   " + ", ".join(cols))
    return rows, cols

_, IRC_COLS = describe("incident_root_cause")
_, KAE_COLS = describe("kpi_avail_enriched")

def guess(cols, wants, kind, avoid=()):
    """Exact name match first, then substring. `avoid` blocks substring hits whose
    name ends with one of those suffixes -- without it, a wants entry of
    'transit_day' matches 'transit_day_key' (an int YYYYMMDD, not a date) purely
    because that column happens to appear first."""
    low = {c.lower(): c for c in cols}
    for w in wants:
        if w.lower() in low:
            return low[w.lower()]
    for w in wants:
        for c in cols:
            if w.lower() in c.lower() and not any(c.lower().endswith(a) for a in avoid):
                return c
    print(f"  !! no {kind} column matched {wants}")
    return None

# SN_U_DEVICE_ID also contains 'device_id' and is only ~3% populated, so the
# exact-match pass above is what keeps us on the real device_id column.
IRC_KEY  = guess(IRC_COLS, ["device_id"], "key")
IRC_DATE = guess(IRC_COLS, ["transit_day", "failure_date", "AE_START_DTM",
                            "start_dtm", "event_dtm"], "date", avoid=("_key",))
KAE_KEY  = guess(KAE_COLS, ["device_id"], "key")
KAE_DATE = guess(KAE_COLS, ["transit_day", "START_DTM", "start_dtm",
                            "event_dtm"], "date", avoid=("_key",))

print(f"\nincident_root_cause : key={IRC_KEY}  date={IRC_DATE}")
print(f"kpi_avail_enriched  : key={KAE_KEY}  date={KAE_DATE}")
print("\nOverride any of these four names in a cell below, then re-run cell 11.")

In [ ]:
# CELL 11 : build device-day lookups and measure how much of the episode fact they cover
def to_day(s):
    """Parse a date column. Handles real timestamps AND integer YYYYMMDD keys --
    pd.to_datetime(20240617) is otherwise read as nanoseconds since the epoch and
    silently returns 1970-01-01 for every row."""
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().mean() > 0.9 and x.dropna().between(19000101, 21001231).mean() > 0.9:
        return pd.to_datetime(x.astype("Int64").astype("string"),
                              format="%Y%m%d", errors="coerce")
    return pd.to_datetime(s, errors="coerce").dt.floor("D")

def daykey(df, keycol, datecol, label):
    d = pd.DataFrame(index=df.index)
    d["_dev"] = df[keycol].astype("string").str.strip().str.upper()
    d["_d"]   = to_day(df[datecol])
    lo, hi = d["_d"].min(), d["_d"].max()
    print(f"  {label:22s} key={keycol:16s} populated {d['_dev'].notna().mean():5.1%}"
          f" | date={datecol:16s} {lo} -> {hi}")
    if pd.notna(hi) and hi.year < 2000:
        print(f"     !! {datecol} parsed to pre-2000 -- wrong column or wrong format")
    return d

ep["_dev"] = ep["device_id"].astype("string").str.strip().str.upper()
ep["_d"]   = ep["episode_start"].dt.floor("D")
EP_IDX     = pd.MultiIndex.from_arrays([ep["_dev"], ep["_d"]])
print("KEY / DATE SANITY -- both sides must span the same era\n")
print(f"  {'episode fact':22s} key=device_id        populated {ep['_dev'].notna().mean():5.1%}"
      f" | date=episode_start   {ep['_d'].min()} -> {ep['_d'].max()}")

# ---- E3 chargeability -------------------------------------------------------
irc = load(f"{SILVER_ROOT}/incident_root_cause")
k = daykey(irc, IRC_KEY, IRC_DATE, "incident_root_cause")
irc_chg = flag(irc["is_chargeable"]) if "is_chargeable" in irc.columns else pd.Series(False, index=irc.index)
irc_day = (pd.DataFrame({"_dev": k["_dev"], "_d": k["_d"], "chg": irc_chg})
             .dropna(subset=["_dev", "_d"]).groupby(["_dev", "_d"])["chg"].max())

ep["_e3_seen"]       = EP_IDX.isin(irc_day.index)
ep["_e3_chargeable"] = pd.Series(irc_day.reindex(EP_IDX).to_numpy(), index=ep.index).eq(True)

# ---- E4 relief --------------------------------------------------------------
kae = load(f"{SILVER_ROOT}/kpi_avail_enriched")
k2 = daykey(kae, KAE_KEY, KAE_DATE, "kpi_avail_enriched")
rel = pd.Series(False, index=kae.index)
if "RELIEF_ID" in kae.columns:
    rel = rel | kae["RELIEF_ID"].notna()
if "EXCLUDED" in kae.columns:
    rel = rel | pd.to_numeric(kae["EXCLUDED"], errors="coerce").fillna(0).gt(0)
kae_day = (pd.DataFrame({"_dev": k2["_dev"], "_d": k2["_d"], "rel": rel})
             .dropna(subset=["_dev", "_d"]).groupby(["_dev", "_d"])["rel"].max())

ep["_e4_seen"]     = EP_IDX.isin(kae_day.index)
ep["_e4_relieved"] = pd.Series(kae_day.reindex(EP_IDX).to_numpy(), index=ep.index).eq(True)

print("\nMATCH RATE -- share of episodes with a same-day row in each source\n")
cov = ep.groupby("mars_device_category")[["_e3_seen", "_e3_chargeable",
                                          "_e4_seen", "_e4_relieved"]].mean()
cov.columns = ["e3 incident seen", "e3 chargeable", "e4 kpi seen", "e4 relieved"]
print(cov.round(4).to_string())
print("\nVALIDATOR at 0.0000 is expected -- ServiceNow does not log bus validators.")
print("GATE or TVM at 0.0000 is a JOIN FAILURE, not a coverage fact: re-read the")
print("key/date sanity block above before believing anything in cell 12.")

In [ ]:
# CELL 12 : measure E3 / E4 on their own and stacked on E1
kpi = pd.Series(False, index=ep.index)
for fl, col in KPI_COL_FOR.items():
    sel, mc = ep["mars_device_category"].eq(fl), f"_mx_{col}"
    if mc in ep.columns:
        kpi.loc[sel] = ep.loc[sel, mc]

chg, relieved = ep["_e3_chargeable"], ep["_e4_relieved"]

V2 = {
    "E0  every OOS Set episode":          pd.Series(True, index=ep.index),
    "E1  counts toward Ventra KPI":       kpi,
    "E3  chargeable":                     chg,
    "E4  NOT relieved / excluded":        ~relieved,
    "E1+E3  KPI-counted AND chargeable":  kpi & chg,
    "E1+E4  KPI-counted, not relieved":   kpi & ~relieved,
    "E1+E3+E4  all three":                kpi & chg & ~relieved,
}

rows = []
for fleet, g in ep.groupby("mars_device_category"):
    ndev = ELIGIBLE_DEVICES.get(fleet)
    if not ndev:
        continue
    device_days = ndev * WIN_DAYS
    for name, mask in V2.items():
        n = int(mask.loc[g.index].sum())
        for h in (3, 7):
            p = min(0.999, n * h / device_days)
            fpr, auc = auc_required(p)
            rows.append(dict(fleet=fleet, variant=name, episodes=n,
                             kept=n / max(1, len(g)), horizon=h, base_rate=p,
                             auc_required=auc, gap=auc - MEASURED_AUC[fleet]))

res2 = pd.DataFrame(rows)
for fleet in sorted(res2["fleet"].unique()):
    print("=" * 100)
    print(f"{fleet}   eligible={ELIGIBLE_DEVICES[fleet]:,}  AUC today={MEASURED_AUC[fleet]}")
    print("=" * 100)
    print(res2[res2["fleet"] == fleet][["variant", "episodes", "kept", "horizon",
          "base_rate", "auc_required", "gap"]].round(4).to_string(index=False))
    print()

res2.to_csv("ps1_label_enrichment_e3_e4.csv", index=False)
print("saved -> ps1_label_enrichment_e3_e4.csv")

d = res2[(res2["horizon"] == 7) & res2["gap"].notna()
         & res2["base_rate"].between(0.005, 0.30) & (res2["episodes"] > 200)]
if len(d):
    print("\nNON-DEGENERATE, adequately populated options at 7 days:")
    print(d.sort_values("gap").groupby("fleet").head(3)
           [["fleet", "variant", "episodes", "base_rate", "auc_required", "gap"]]
           .round(4).to_string(index=False))
print("\nAny variant whose episode count collapses to near zero is reporting the source's")
print("coverage, not a property of the label. Check cell 11's match rate before quoting it.")